# P09 Воспроизводимый эксперимент

**Рабочий ноутбук**. Полное объяснение: [локальный урок P09](../../course/pilot/P09.md). Этот файл содержит исполняемый практикум.

Начните с [порядка самостоятельной работы](../../course/self-study.md). Локальный учебник открывается файлом START_HERE.html в корне курса. Если ноутбук скопирован в submissions и относительная ссылка не открывается, найдите урок по пути course/pilot/P09.md от корня репозитория.

## Цель

Разделить train/validation/test, выбрать модель по validation и сравнить с baseline.

**Перед началом:** P03 о независимой проверке, P08 о параметрах; новый термин baseline — простое исходное решение.

## Подготовка

Запускайте из корня репозитория в его виртуальном окружении. Первичная установка зависимостей описана в README. После установки выполнение не использует сеть, платные API или GPU.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mlcourse" / "pilot.py").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("Откройте Jupyter из корня ml-llm-course")
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from mlcourse.pilot import (
    mse, predict_line, linear_gradient, central_difference, fit_standardizer,
    matvec, fit_gradient, make_dataset, run_experiment, predict_payload,
)
plt.rcParams.update({"figure.figsize": (7.2, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
FIGURES = ROOT / "artifacts" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Данные: искусственные учебные примеры. CPU. Состояние ученика не оценивается автоматически.")

## Разобранный пример

100 синтетических примеров имеют известный генератор, но добавленный шум не позволяет получить нулевую ошибку простой линией. Генератор и seed фиксируют повторяемость. Модель выбирается по validation; test используется после выбора. Результат относится только к этой учебной постановке.

### Прочитать фиксированные данные

In [ ]:
frame = pd.read_csv(ROOT / "data" / "pilot" / "regression.csv")
display(frame.head(6).round(4))
print(frame.groupby("split").size().to_dict())
assert len(frame) == 100
assert frame["row_id"].is_unique
assert frame.groupby("split").size().to_dict() == {"test":20, "train":60, "validation":20}

### Обучить исходное и линейное решения

In [ ]:
from sklearn.linear_model import LinearRegression
train = frame[frame["split"] == "train"]
validation = frame[frame["split"] == "validation"]
test = frame[frame["split"] == "test"]
baseline = train["y"].mean()
model = LinearRegression().fit(train[["x"]], train["y"])
validation_scores = {"baseline": mse(validation["y"], np.full(len(validation), baseline)),
                     "linear": mse(validation["y"], model.predict(validation[["x"]]))}
chosen = min(validation_scores, key=validation_scores.get)
test_prediction = (model.predict(test[["x"]]) if chosen == "linear" else np.full(len(test), baseline))
report = run_experiment()
assert chosen == report["selected_on_validation"]
assert np.isclose(mse(test["y"], test_prediction), report["test_mse"], atol=1e-10)
print(json.dumps(report, ensure_ascii=False, indent=2))

### Посмотреть данные и ошибки

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, color, marker in [("train", "#1f5a99", "o"), ("validation", "#b56c19", "s"), ("test", "#555555", "x")]:
    subset = frame[frame["split"] == name]
    axes[0].scatter(subset["x"], subset["y"], s=24, color=color, marker=marker, label=name)
grid = np.linspace(-2, 2, 100)
axes[0].plot(grid, model.coef_[0]*grid + model.intercept_, color="#333333", linestyle="--", label="Модель")
axes[0].set(xlabel="x, условные единицы", ylabel="y, условные единицы", title="Синтетические данные и модель")
axes[0].legend(fontsize=9)
axes[1].scatter(test_prediction, test["y"].to_numpy()-test_prediction, color="#1f5a99", s=25)
axes[1].axhline(0, color="#555555", linestyle="--")
axes[1].set(xlabel="Предсказание", ylabel="y − предсказание", title="Ошибки на 20 тестовых примерах")
fig.tight_layout(); fig.savefig(FIGURES / "P09_experiment.png", dpi=130); plt.show()

## Практика с поддержкой

Назовите строку, где выбирается модель. Покажите, что в этой строке нет test. Сравните отчёт с повторным запуском.

## Самостоятельная задача

Повторите эксперимент на новой синтетической задаче с noise=0.7 и тем же seed. Сначала зафиксируйте план выбора. Сравните baseline и линейную модель по validation, затем один раз измерьте test. Разберите три самых крупных ошибки и объясните, почему нельзя перенастраивать модель после просмотра test.

Сначала сохраните собственную попытку. Затем откройте подсказки в локальном уроке и сверяйтесь с эталоном; отмечайте, какая помощь потребовалась. После полного разбора выполните новый вариант без подсказок. При желании можно передать работу на разбор.

In [ ]:
# Здесь ваша реализация или расчёт.
# Ячейка намеренно не содержит готового решения и не ломает запуск примеров.

**Ваше объяснение:** замените этот абзац своим рассуждением, результатом и ограничением. Это поле для ответа ученика, а не пропущенное объяснение автора.

## Проверка

Разделение 60/20/20 неизменно; baseline учится на train; выбор по validation; test не влияет на выбор; вывод ограничен данной задачей.

Успешный Run All подтверждает запуск подготовленных примеров. Для самопроверки сравните свою работу с критериями и эталоном, объясните шаги своими словами и решите новый вариант. Запишите результат в progress/progress.md относительно корня курса. Самооценка не является независимым подтверждением навыка.

## Следующий шаг

Предложите способ получить более устойчивую оценку без многократной подгонки к одному test. Объясните отличие проверки гипотезы от её генерации по тестовым ошибкам.

[Основной источник](https://scikit-learn.org/stable/common_pitfalls.html). Все значения в примерах синтетические. Сохраняйте исходные ноутбуки; для своей работы используйте копию.